# Document Image Augmentation and Classification

This notebook builds a small synthetic document image task:
- Class 0: receipt-like layout
- Class 1: invoice-like layout

A/B test:
- baseline
- augmented with rotation, blur, and contrast change

This is useful for document-image classification ideas.

In [ ]:
# Pillow is already available in most Colab environments.
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)

In [ ]:
def make_doc_image(label, size=(128, 128)):
    img = Image.new("L", size, color=255)
    draw = ImageDraw.Draw(img)

    # Simple header
    if label == 0:
        title = "RECEIPT"
        rows = ["Item A   5.00", "Item B   7.00", "Tax      1.20", "Total   13.20"]
    else:
        title = "INVOICE"
        rows = ["Client XYZ", "Service  50.00", "Fee      12.00", "Total    62.00"]

    draw.text((10, 10), title, fill=0)
    y = 35
    for row in rows:
        draw.text((10, y), row, fill=0)
        y += 18

    # random lines and boxes
    draw.rectangle((8, 8, 118, 110), outline=0, width=1)
    return img

def augment_doc(img):
    img = img.rotate(np.random.uniform(-8, 8), fillcolor=255)
    if np.random.rand() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(radius=1))
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(np.random.uniform(0.8, 1.2))
    return img

def build_dataset(n=1000, augment=False):
    X, y = [], []
    for i in range(n):
        label = i % 2
        img = make_doc_image(label)
        if augment:
            img = augment_doc(img)
        arr = np.array(img, dtype=np.float32) / 255.0
        X.append(arr[..., None])
        y.append(label)
    return np.array(X), np.array(y)

X_train, y_train = build_dataset(1000, augment=False)
X_test, y_test = build_dataset(300, augment=False)

X_aug_extra, y_aug_extra = build_dataset(500, augment=True)
X_train_aug = np.concatenate([X_train, X_aug_extra], axis=0)
y_train_aug = np.concatenate([y_train, y_aug_extra], axis=0)

In [ ]:
def build_doc_model():
    model = keras.Sequential([
        layers.Input(shape=(128, 128, 1)),
        layers.Conv2D(16, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

base_model = build_doc_model()
base_model.fit(X_train, y_train, epochs=4, batch_size=32, verbose=0)
base_acc = base_model.evaluate(X_test, y_test, verbose=0)[1]

aug_model = build_doc_model()
aug_model.fit(X_train_aug, y_train_aug, epochs=4, batch_size=32, verbose=0)
aug_acc = aug_model.evaluate(X_test, y_test, verbose=0)[1]

print("Baseline accuracy:", round(base_acc, 4))
print("Augmented accuracy:", round(aug_acc, 4))